In [ ]:
import os
import numpy as np

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize LLM Client
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# 2. Knowledge Base
documents = [
    {
        "id": "HR-001",
        "title": "Annual Leave Policy",
        "text": "Employees receive 26 annual leave days.",
    },
    {
        "id": "HR-002",
        "title": "Leave Approval Policy",
        "text": "Leave requests must be approved by the manager.",
    },
    {
        "id": "HR-003",
        "title": "Health Insurance Policy",
        "text": "Employees have supplementary health insurance.",
    },
    {
        "id": "HR-004",
        "title": "Salary Policy",
        "text": "Monthly salaries are paid at the end of each month.",
    },
    {
        "id": "HR-005",
        "title": "Remote Work Policy",
        "text": "Employees can work remotely two days per week.",
    },
]

# 3. Load Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

document_embeddings = embedding_model.encode(
    [doc["text"] for doc in documents], convert_to_numpy=True, normalize_embeddings=True
)


# 4. Retrieve Relevant Chunks
def retrieve_chunks(query, top_k=3, threshold=0.35):

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices:
        score = float(scores[index])

        if score < threshold:
            continue

        results.append(
            {
                "id": documents[index]["id"],
                "title": documents[index]["title"],
                "text": documents[index]["text"],
                "score": score,
            }
        )

        if len(results) >= top_k:
            break

    return results


# 5. Construct Context
def construct_context(chunks):

    context_parts = []

    for chunk in chunks:
        context_parts.append(
            f"[Source ID: {chunk['id']}]\n[Document: {chunk['title']}]\n{chunk['text']}"
        )

    return "\n\n".join(context_parts)


# 6. Generate Answer
def generate_answer(query, context):

    if not context.strip():
        return "I couldn't find relevant information in the documents."

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": """
You are an HR assistant.

Answer using only the provided context.
Do not invent information.
If the answer is unavailable, say so.
Do not create or guess source IDs.
""",
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{query}
""",
            },
        ],
    )

    return response.choices[0].message.content


# 7. Execute RAG Pipeline
def ask_hr_assistant(query):

    retrieved_chunks = retrieve_chunks(query)

    context = construct_context(retrieved_chunks)

    answer = generate_answer(query, context)

    sources = [
        {"id": chunk["id"], "title": chunk["title"], "score": chunk["score"]}
        for chunk in retrieved_chunks
    ]

    return {"question": query, "answer": answer, "sources": sources}


# 8. Test
result = ask_hr_assistant("How many annual leave days do employees receive?")

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:
    print(f"- {source['id']}: {source['title']}")
    print(f"  Similarity: {source['score']:.4f}")